# Notebook 6 — Clasificador SimpleRNN (Baseline) → MongoDB
### Investing AI · iDeSo · UNMSM

**Objetivo:** Leer datos de `precios_ohlcv` en MongoDB, construir secuencias temporales,
entrenar un clasificador **SimpleRNN** (Red Neuronal Recurrente Simple, sin compuertas) para
predecir señales BUY/SELL, y usarlo como **baseline** para comparar contra arquitecturas más
avanzadas con compuertas (LSTM, BiLSTM, GRU) ya entrenadas en notebooks anteriores.

**Arquitectura:** `SimpleRNN(180, return_sequences=True) → SimpleRNN(90) → Dense(1, sigmoid)`
**Entrenamiento:** 80 épocas · batch_size=64 · Adam(lr=0.001) · binary_crossentropy

**Prerequisito:** Haber ejecutado el Notebook 1 (colección `precios_ohlcv` poblada) y,
para la comparación final, el Notebook 3 (`modelo='LSTM'`).
**Salida:** Colecciones `predicciones` (`modelo='SimpleRNN'`) y `metricas_modelos`
(`modelo='SimpleRNN'`) listas para la API, además de un archivo `datos_simplernn_clf.json`
exportado para el frontend.

> **¿Por qué un baseline?** El `SimpleRNN` no tiene compuertas de olvido/actualización como el
> LSTM o el GRU, por lo que sufre más el problema del *vanishing gradient* en secuencias largas.
> Sirve como punto de referencia mínimo: si LSTM/BiLSTM/GRU no lo superan con un margen claro,
> eso sugiere que la complejidad adicional de esas arquitecturas no se está traduciendo en mejor
> desempeño para este problema y estos datos.


## Paso 1 — Instalación de librerías

In [ ]:
# Instalación de librerías necesarias
# TensorFlow ya viene preinstalado en Colab; se asegura scikit-learn y pymongo
!pip install 'pymongo[srv]' scikit-learn --quiet
print('✓ Librerías instaladas')

## Paso 2 — Conexión a MongoDB Atlas

In [ ]:
# Conectar a MongoDB Atlas usando el Secret MONGO_URI de Colab
# (Secrets: ícono de llave en el panel izquierdo de Colab)
from google.colab import userdata
from pymongo import MongoClient
import pandas as pd
import numpy as np
from datetime import datetime
import json
import random

MONGO_URI = userdata.get('MONGO_URI')
client    = MongoClient(MONGO_URI)
db        = client['spbi']

# Verificar conexión
client.admin.command('ping')
print('✓ Conectado a MongoDB Atlas')
print(f'  Base de datos: spbi')
print(f'  Colecciones disponibles: {db.list_collection_names()}')

## Paso 3 — Semilla fija y configuración de reproducibilidad

In [ ]:
# SEED fija para reproducibilidad (numpy, python, tensorflow)
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✓ Semilla fija SEED={SEED} aplicada (random, numpy, tensorflow)')
print(f'  TensorFlow versión: {tf.__version__}')

## Paso 4 — Carga de datos desde MongoDB

In [ ]:
# Función para cargar precios OHLCV de un ticker desde MongoDB
def cargar_datos(ticker):
    """
    Lee los documentos de precios_ohlcv para un ticker
    y los devuelve como DataFrame ordenado por fecha.
    """
    docs = list(db['precios_ohlcv'].find({'ticker': ticker}).sort('fecha', 1))
    if not docs:
        return pd.DataFrame()
    df = pd.DataFrame(docs)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)
    return df

# Verificación rápida con BVN
df_test = cargar_datos('BVN')
print(f'✓ BVN: {len(df_test)} registros cargados')
print(df_test[['fecha','close','sma_20','rsi_14']].tail(3).to_string(index=False))

## Paso 5 — Ingeniería de features, secuencias temporales y target

Se reutiliza exactamente el mismo feature engineering, target BUY/SELL y esquema de secuencias
del clasificador LSTM (Notebook 3), para que la comparación entre arquitecturas sea **justa**:
mismos datos, mismas features, mismo `N_STEPS`, mismo target. La única diferencia entre este
notebook y el Notebook 3 es la arquitectura de la red (SimpleRNN en vez de LSTM).


In [ ]:
from sklearn.preprocessing import MinMaxScaler

N_STEPS = 20  # tamaño de la ventana temporal (días) — igual que en el Notebook 3 (LSTM)

# Features para tickers con precio variable (FSM, ABX.TO, BVN, BHP)
FEATURES_PRECIO = ['close','sma_20','sma_50','ema_12','ema_26','rsi_14','retorno']

# Features especiales para VOLCABC1.LM (precio constante → usar volumen)
FEATURES_VOLUMEN = ['vol_change_pct','vol_relative','vol_rsi','obv_slope','money_flow_rel']

def preparar_features_precio(df):
    """
    Feature engineering estándar basado en precio e indicadores técnicos.
    Target: 1 (BUY) si el cierre del día siguiente es mayor, 0 (SELL) si baja.
    """
    df = df.copy()
    df['retorno'] = df['close'].pct_change()
    # Target: ¿sube mañana?
    df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
    df = df.dropna(subset=FEATURES_PRECIO + ['target'])

    feats   = df[FEATURES_PRECIO].values
    target  = df['target'].values
    fechas  = df['fecha'].values
    precios = df['close'].values
    return feats, target, fechas, precios

def preparar_features_volumen(df):
    """
    Feature engineering basado en VOLUMEN para VOLCABC1.LM.
    Usado porque el precio de ese ticker es casi constante y los
    indicadores técnicos estándar no aportan señal estadística útil.
    """
    df = df.copy()
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce').fillna(0)

    # Cambio porcentual diario de volumen
    df['vol_change_pct'] = df['volume'].pct_change().fillna(0).clip(-5, 5)
    # Volumen relativo respecto a su media de 20 días
    df['vol_sma20']      = df['volume'].rolling(20).mean()
    df['vol_relative']   = (df['volume'] / df['vol_sma20'].replace(0, np.nan)).fillna(1).clip(0, 10)
    # OBV (On-Balance Volume)
    direction    = np.sign(df['close'].diff().fillna(0))
    df['obv']    = (direction * df['volume']).cumsum()
    df['obv_sma20']  = df['obv'].rolling(20).mean()
    df['obv_slope']  = df['obv'].diff(5).fillna(0)
    # RSI aplicado al volumen
    delta_vol = df['volume'].diff()
    gain_vol  = delta_vol.clip(lower=0).rolling(14).mean()
    loss_vol  = (-delta_vol.clip(upper=0)).rolling(14).mean()
    rs_vol    = gain_vol / loss_vol.replace(0, np.nan)
    df['vol_rsi'] = (100 - 100 / (1 + rs_vol)).fillna(50)
    # Money flow relativo
    df['money_flow']     = df['close'] * df['volume']
    df['mf_sma10']       = df['money_flow'].rolling(10).mean()
    df['money_flow_rel'] = (df['money_flow'] / df['mf_sma10'].replace(0, np.nan)).fillna(1)

    # Target: señal basada en OBV vs su media móvil (sin filtro de magnitud)
    # Nota: igual que en el Notebook 3, NO se descartan filas por magnitud de
    # 'vol_relative', porque la red recurrente necesita tramos de días
    # CONSECUTIVOS para poder armar secuencias temporales.
    df['target'] = (df['obv'] > df['obv_sma20']).astype(int)
    df = df.dropna(subset=FEATURES_VOLUMEN)

    feats   = df[FEATURES_VOLUMEN].values
    target  = df['target'].values
    fechas  = df['fecha'].values
    precios = df['close'].values
    return feats, target, fechas, precios

def crear_secuencias(feats, target, fechas, precios, n_steps=N_STEPS):
    """
    Convierte una matriz de features 2D (muestras, features) en
    secuencias 3D (muestras, n_steps, features) mediante ventana deslizante.
    El target de cada secuencia es el target del último día de la ventana.
    """
    X_seq, y_seq, fechas_seq, precios_seq = [], [], [], []
    for i in range(n_steps, len(feats)):
        X_seq.append(feats[i - n_steps:i])
        y_seq.append(target[i])
        fechas_seq.append(fechas[i])
        precios_seq.append(precios[i])
    return (np.array(X_seq), np.array(y_seq),
            np.array(fechas_seq), np.array(precios_seq))

print(f'✓ Funciones de feature engineering y secuencias definidas (N_STEPS={N_STEPS})')

## Paso 6 — Construcción del modelo SimpleRNN (Keras Sequential)

A diferencia del LSTM, el `SimpleRNN` solo tiene un estado oculto que se recalcula en cada paso
con una única transformación lineal + activación (`tanh` por defecto), sin compuertas de
olvido/entrada/salida. Esto lo hace más rápido de entrenar por época, pero más propenso a perder
información de largo plazo dentro de la ventana de `N_STEPS` días.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

def construir_modelo_simplernn(n_steps, n_features):
    """
    Arquitectura SimpleRNN (baseline) para clasificación binaria BUY/SELL:
    SimpleRNN(180, return_sequences=True) -> SimpleRNN(90) -> Dense(1, sigmoid)
    Nota: no se agrega Dropout entre capas para mantener la comparación con
    la especificación exacta del baseline; si se observa sobreajuste fuerte,
    EarlyStopping sobre val_loss actúa como principal control de regularización.
    """
    modelo = Sequential([
        Input(shape=(n_steps, n_features)),
        SimpleRNN(180, return_sequences=True),
        SimpleRNN(90),
        Dense(1, activation='sigmoid')
    ])

    modelo.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

print('✓ Función construir_modelo_simplernn definida')
print('  Arquitectura: SimpleRNN(180, return_sequences=True) -> SimpleRNN(90) -> Dense(1, sigmoid)')

## Paso 7 — Entrenamiento y evaluación del SimpleRNN

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix)

def entrenar_simplernn(X, y):
    """
    Entrena el SimpleRNN sobre una partición temporal 80/20 (sin data leakage).
    - MinMaxScaler ajustado SOLO con datos de entrenamiento (aplicado feature a feature)
    - 80 épocas, batch_size=64, Adam(lr=0.001)
    - class_weight balanceado (igual que en el LSTM, para comparación justa)
    - EarlyStopping sobre val_loss para evitar sobreajuste excesivo
    """
    n     = len(X)
    corte = int(n * 0.80)

    n_steps, n_features = X.shape[1], X.shape[2]

    # Escalado: el scaler se ajusta sobre los datos de train aplanados (2D)
    scaler = MinMaxScaler()
    X_train_flat = X[:corte].reshape(-1, n_features)
    scaler.fit(X_train_flat)

    def escalar(X_):
        forma = X_.shape
        X_flat = X_.reshape(-1, n_features)
        X_esc  = scaler.transform(X_flat)
        return X_esc.reshape(forma)

    X_train, X_test = escalar(X[:corte]), escalar(X[corte:])
    y_train, y_test = y[:corte], y[corte:]

    # Pesos de clase para corregir desbalance (equivalente a class_weight='balanced')
    clases, conteos = np.unique(y_train, return_counts=True)
    total = len(y_train)
    pesos_clase = {int(c): total / (len(clases) * cnt) for c, cnt in zip(clases, conteos)}

    modelo = construir_modelo_simplernn(n_steps, n_features)

    early_stop = EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=0
    )

    historial = modelo.fit(
        X_train, y_train,
        epochs=80,
        batch_size=64,
        validation_split=0.15,
        class_weight=pesos_clase,
        callbacks=[early_stop],
        verbose=0
    )

    y_prob = modelo.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    metricas = {
        'accuracy' : round(float(accuracy_score(y_test,  y_pred)), 4),
        'precision': round(float(precision_score(y_test, y_pred, average='weighted', zero_division=0)), 4),
        'recall'   : round(float(recall_score(y_test,    y_pred, average='weighted', zero_division=0)), 4),
        'f1'       : round(float(f1_score(y_test,        y_pred, average='weighted', zero_division=0)), 4),
        'epocas_entrenadas': int(len(historial.history['loss'])),
        'n_train'  : int(len(X_train)),
        'n_test'   : int(len(X_test))
    }

    cm = confusion_matrix(y_test, y_pred).tolist()

    hist_epocas = {
        'loss'         : [round(float(v), 4) for v in historial.history['loss']],
        'accuracy'     : [round(float(v), 4) for v in historial.history['accuracy']],
        'val_loss'     : [round(float(v), 4) for v in historial.history.get('val_loss', [])],
        'val_accuracy' : [round(float(v), 4) for v in historial.history.get('val_accuracy', [])]
    }

    return modelo, scaler, metricas, cm, hist_epocas, X_test, y_test

print('✓ Función entrenar_simplernn definida')
print('  - class_weight balanceado (evita F1=0.000 por desbalance)')
print('  - MinMaxScaler ajustado solo con train (sin leakage)')
print('  - EarlyStopping sobre val_loss, patience=10')

## Paso 8 — Pipeline completo por ticker

In [ ]:
def procesar_ticker(ticker):
    """
    Pipeline completo para un ticker:
    1. Carga datos de MongoDB
    2. Selecciona feature engineering (precio o volumen según ticker)
    3. Construye secuencias temporales de tamaño N_STEPS
    4. Entrena SimpleRNN sobre partición temporal 80/20
    5. Genera historial de señales sobre todas las secuencias
    6. Guarda predicción actual + métricas + historial en MongoDB
    """
    print(f'\n  [{ticker}] Procesando...')

    df = cargar_datos(ticker)
    if len(df) < 100:
        print(f'  [{ticker}] ✗ Datos insuficientes ({len(df)} registros) — se omite')
        return None

    # VOLCABC1.LM usa features de volumen (precio casi constante)
    if ticker == 'VOLCABC1.LM':
        feats, target, fechas, precios = preparar_features_volumen(df)
        tipo_features = 'volumen'
    else:
        feats, target, fechas, precios = preparar_features_precio(df)
        tipo_features = 'precio'

    X, y, fechas_seq, precios_seq = crear_secuencias(feats, target, fechas, precios, N_STEPS)

    if len(X) < 50:
        print(f'  [{ticker}] ✗ Secuencias insuficientes tras limpieza — se omite')
        return None

    print(f'  [{ticker}] Features: {tipo_features} | Secuencias: {len(X)} | N_STEPS={N_STEPS}')

    modelo, scaler, metricas, cm, hist_epocas, X_test, y_test = entrenar_simplernn(X, y)

    # ── Señal actual (última secuencia disponible) ──────────────
    n_features   = X.shape[2]
    X_esc        = scaler.transform(X.reshape(-1, n_features)).reshape(X.shape)
    ultima_X     = X_esc[-1:]
    prob_actual  = float(modelo.predict(ultima_X, verbose=0)[0][0])
    pred_actual  = int(prob_actual >= 0.5)
    senal_actual = 'BUY' if pred_actual == 1 else 'SELL'

    # ── Historial de señales (todas las secuencias) ──────────────
    probas_all = modelo.predict(X_esc, verbose=0).flatten()
    preds_all  = (probas_all >= 0.5).astype(int)

    historico_senales = [
        {
            'fecha'       : str(fechas_seq[i])[:10],
            'precio'      : round(float(precios_seq[i]), 4),
            'prediccion'  : 'BUY' if preds_all[i] == 1 else 'SELL',
            'probabilidad': round(float(probas_all[i]), 4)
        }
        for i in range(len(preds_all))
    ]

    # ── Guardar predicción actual en MongoDB ─────────────────────
    db['predicciones'].delete_many({'ticker': ticker, 'modelo': 'SimpleRNN'})
    db['predicciones'].insert_one({
        'ticker'           : ticker,
        'modelo'           : 'SimpleRNN',
        'senal'            : senal_actual,
        'confianza'        : round(prob_actual if pred_actual == 1 else 1 - prob_actual, 4),
        'fecha_prediccion' : datetime.now().strftime('%Y-%m-%d'),
        'historico_senales': historico_senales,
        'tipo_features'    : tipo_features,
        'n_steps'          : N_STEPS,
        'created_at'       : datetime.now()
    })

    # ── Guardar métricas en MongoDB ───────────────────────────────
    db['metricas_modelos'].delete_many({'ticker': ticker, 'modelo': 'SimpleRNN'})
    db['metricas_modelos'].insert_one({
        'ticker'             : ticker,
        'modelo'             : 'SimpleRNN',
        **metricas,
        'matriz_confusion'   : cm,
        'historial_epocas'   : hist_epocas,
        'tipo_features'      : tipo_features,
        'n_steps'            : N_STEPS,
        'fecha_entrenamiento': datetime.now()
    })

    a  = metricas['accuracy']
    f1 = metricas['f1']
    ep = metricas['epocas_entrenadas']
    print(f'  [{ticker}] ✓ senal={senal_actual} ({max(prob_actual, 1-prob_actual):.0%}) | acc={a:.0%} | f1={f1:.0%} | épocas={ep}')

    return {
        'ticker': ticker, 'tipo_features': tipo_features,
        'senal': senal_actual, 'metricas': metricas,
        'matriz_confusion': cm, 'historial_epocas': hist_epocas,
        'historico_senales': historico_senales
    }

print('✓ Función procesar_ticker definida')

## Paso 9 — Ejecutar pipeline para los 5 tickers

In [ ]:
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

print('=' * 55)
print('  SimpleRNN (BASELINE) — ENTRENAMIENTO PARA 5 TICKERS')
print('=' * 55)

resultados = {}
for ticker in TICKERS:
    try:
        res = procesar_ticker(ticker)
        if res is not None:
            resultados[ticker] = res
    except Exception as e:
        print(f'  [{ticker}] ✗ Error inesperado: {e}')

print()
print('=' * 55)
print(f'  ✅ Pipeline completo: {len(resultados)}/{len(TICKERS)} tickers procesados')
print('=' * 55)

## Paso 10 — Verificación en MongoDB

In [ ]:
print('PREDICCIONES SimpleRNN en MongoDB')
print('-' * 45)
for doc in db['predicciones'].find({'modelo': 'SimpleRNN'}, {'_id':0,'historico_senales':0,'created_at':0}):
    t  = doc['ticker']
    s  = doc['senal']
    c  = doc['confianza']
    tf = doc.get('tipo_features','precio')
    print(f'  {t:<15} {s:<5} ({c:.0%}) | features: {tf}')

print()
print('MÉTRICAS SimpleRNN en MongoDB')
print('-' * 45)
for doc in db['metricas_modelos'].find({'modelo': 'SimpleRNN'}, {'_id':0,'matriz_confusion':0,'historial_epocas':0,'fecha_entrenamiento':0}):
    t  = doc['ticker']
    a  = doc['accuracy']
    f1 = doc['f1']
    p  = doc['precision']
    r  = doc['recall']
    ep = doc.get('epocas_entrenadas','?')
    print(f'  {t:<15} acc={a:.0%} | f1={f1:.0%} | prec={p:.0%} | rec={r:.0%} | épocas={ep}')

print()
print('✅ Colecciones predicciones y metricas_modelos (SimpleRNN) verificadas')

## Paso 11 — Comparación contra arquitecturas con compuertas (LSTM / BiLSTM / GRU)

Se comparan las métricas del `SimpleRNN` (baseline) contra las de cualquier otra arquitectura ya
guardada en `metricas_modelos` con `modelo` en `{'LSTM', 'BiLSTM', 'GRU'}`. La comparación usa
**F1-score ponderado** como criterio principal (más robusto que accuracy ante el desbalance de
clases BUY/SELL), y también reporta la diferencia en accuracy.

**Criterio de "mejora significativa":** se considera que una arquitectura con compuertas supera
de forma significativa al baseline si su F1 promedio (sobre los 5 tickers) es **al menos 3 puntos
porcentuales mayor** que el del SimpleRNN. Este umbral es una convención pedagógica razonable
para trabajar con solo 5 tickers y una sola corrida por modelo (no hay repeticiones ni test
estadístico formal de significancia).


In [ ]:
ARQUITECTURAS_COMPARAR = ['LSTM', 'BiLSTM', 'GRU']
UMBRAL_MEJORA_F1 = 0.03  # 3 puntos porcentuales

def obtener_metricas_promedio(nombre_modelo):
    """
    Lee metricas_modelos para un nombre de modelo dado y devuelve el
    promedio de accuracy/f1/precision/recall sobre los tickers disponibles.
    Devuelve None si no hay documentos para ese modelo.
    """
    docs = list(db['metricas_modelos'].find({'modelo': nombre_modelo}, {'_id': 0}))
    if not docs:
        return None
    return {
        'n_tickers': len(docs),
        'accuracy' : round(float(np.mean([d['accuracy']  for d in docs])), 4),
        'f1'       : round(float(np.mean([d['f1']        for d in docs])), 4),
        'precision': round(float(np.mean([d['precision'] for d in docs])), 4),
        'recall'   : round(float(np.mean([d['recall']    for d in docs])), 4),
    }

baseline = obtener_metricas_promedio('SimpleRNN')
print('BASELINE — SimpleRNN')
print('-' * 60)
if baseline:
    print(f"  Tickers: {baseline['n_tickers']} | acc={baseline['accuracy']:.1%} | "
          f"f1={baseline['f1']:.1%} | prec={baseline['precision']:.1%} | rec={baseline['recall']:.1%}")
else:
    print('  ✗ No hay métricas de SimpleRNN en MongoDB. Ejecuta primero el Paso 9.')

print()
print('COMPARACIÓN CONTRA ARQUITECTURAS CON COMPUERTAS')
print('-' * 60)

comparacion = {'baseline_simplernn': baseline, 'arquitecturas': {}}

for nombre in ARQUITECTURAS_COMPARAR:
    metricas_modelo = obtener_metricas_promedio(nombre)
    if metricas_modelo is None:
        print(f'  {nombre:<10} — no encontrado en MongoDB (aún no fue entrenado/guardado)')
        comparacion['arquitecturas'][nombre] = None
        continue

    if baseline:
        delta_f1  = round(metricas_modelo['f1'] - baseline['f1'], 4)
        delta_acc = round(metricas_modelo['accuracy'] - baseline['accuracy'], 4)
        supera_significativamente = delta_f1 >= UMBRAL_MEJORA_F1
        veredicto = '✓ SUPERA al baseline (>= 3pp F1)' if supera_significativamente else '≈ NO supera de forma significativa'
    else:
        delta_f1 = delta_acc = None
        veredicto = '(sin baseline para comparar)'

    comparacion['arquitecturas'][nombre] = {
        'metricas': metricas_modelo,
        'delta_f1_vs_simplernn' : delta_f1,
        'delta_accuracy_vs_simplernn': delta_acc,
        'supera_significativamente': (delta_f1 >= UMBRAL_MEJORA_F1) if delta_f1 is not None else None
    }

    print(f"  {nombre:<10} acc={metricas_modelo['accuracy']:.1%} | f1={metricas_modelo['f1']:.1%} "
          f"| Δf1={delta_f1:+.1%} | {veredicto}" if delta_f1 is not None else f'  {nombre:<10} {veredicto}')

print()
print('✅ Comparación completada. Ver diccionario `comparacion` para el detalle programático.')

## Paso 12 — Exportación a JSON para el frontend (`datos_simplernn_clf.json`)

In [ ]:
# Estructura de exportación pensada para el módulo comparativo de arquitecturas RNN del frontend
export_data = {
    'modelo'        : 'SimpleRNN',
    'arquitectura'  : 'SimpleRNN(180, return_sequences=True) -> SimpleRNN(90) -> Dense(1, sigmoid)',
    'rol'           : 'baseline',
    'n_steps'       : N_STEPS,
    'seed'          : SEED,
    'fecha_export'  : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'tickers'       : {},
    'comparacion_arquitecturas': comparacion
}

for ticker, res in resultados.items():
    export_data['tickers'][ticker] = {
        'senal'             : res['senal'],
        'tipo_features'     : res['tipo_features'],
        'metricas'          : res['metricas'],
        'matriz_confusion'  : res['matriz_confusion'],
        'historial_epocas'  : res['historial_epocas'],
        'historico_senales' : res['historico_senales'][-60:]  # últimos 60 días para no saturar el JSON
    }

with open('datos_simplernn_clf.json', 'w', encoding='utf-8') as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print(f'✓ Archivo datos_simplernn_clf.json exportado ({len(resultados)} tickers)')
print(f'  Tamaño aproximado: {len(json.dumps(export_data)) / 1024:.1f} KB')

# Descargar el archivo desde Colab (opcional)
try:
    from google.colab import files
    files.download('datos_simplernn_clf.json')
    print('✓ Descarga iniciada en el navegador')
except Exception as e:
    print(f'  (Descarga automática no disponible en este entorno: {e})')

## Resultado

Las colecciones `predicciones` y `metricas_modelos` contienen resultados reales del SimpleRNN
(`modelo='SimpleRNN'`), usado como **baseline** de comparación:
- **5 tickers** procesados con el mismo feature engineering, `N_STEPS` y target que el LSTM
  (Notebook 3), para que la comparación entre arquitecturas sea justa.
- **class_weight balanceado** aplicado → mismo tratamiento del desbalance que en LSTM/BiLSTM/GRU.
- **Sin data leakage** → partición temporal 80/20 estricta, MinMaxScaler ajustado solo con train.
- **SEED=42** fija → resultados reproducibles.
- **Comparación automática** contra LSTM/BiLSTM/GRU (si ya están en MongoDB) usando un umbral de
  mejora de F1 de 3 puntos porcentuales, guardada en `datos_simplernn_clf.json` bajo la clave
  `comparacion_arquitecturas`.
- **datos_simplernn_clf.json** exportado para consumo directo del módulo comparativo del frontend.

**La API (Notebook 4)** puede leer estas colecciones (filtrando `modelo='SimpleRNN'`) y servirlas
al frontend vía HTTP, en paralelo a las predicciones del SVC, LSTM, BiLSTM y GRU.

**Pipeline completo:** `precios_ohlcv` → SimpleRNN → `predicciones` + `metricas_modelos` +
`datos_simplernn_clf.json` (+ comparación vs. arquitecturas con compuertas) ✓
